# Week 3 — Data Contract & Feature Leakage Check

## 1. Data Contract

### 1. What does one row represent?
One row represents one content item/page for a specific client and reporting date in the warehouse's daily content-performance table.

### 2. Which table(s) will I use?
I will use:
- `fact_content_daily_performance` for daily content-level search performance.
- `dim_content` for content metadata and attributes.
- `dim_clients` when client-level information is required.

### 3. What time window will I use?
I will develop and verify the analysis using the mid-panel month **March 2026**. I will avoid using the final month (June 2026) while developing the label logic because it represents the natural future/outcome window.

### 4. What will I predict or rank?
I will explore whether historical content and search-performance signals can identify content pages that are candidates for a potential refresh. For the initial experiment, the target/proxy will be a performance-decline signal rather than a claim that a refresh will definitely improve the page.

### 5. What will I deliberately exclude?
I will deliberately exclude information that is derived from the outcome window or directly encodes the target, such as future performance metrics and label-derived fields, to avoid data leakage.

In [ ]:
%pip -q install duckdb huggingface_hub pandas

In [1]:
import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
            from google.colab import userdata
            HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
                                pass

                                if not HF_TOKEN:
                                    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

                                    print("Hugging Face token loaded successfully.")

In [4]:
import duckdb
import pandas as pd

con = duckdb.connect()

con.execute(
    f"""
        CREATE OR REPLACE SECRET hf
            (
                    TYPE huggingface,
                            TOKEN '{HF_TOKEN}'
                                )
                                    """
                                    )

print("DuckDB connected successfully.")

DuckDB connected successfully.


In [5]:
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
        "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
            "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
                "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
                    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
                    }

print("Warehouse tables configured successfully.")

Warehouse tables configured successfully.


In [ ]:
for name, src in TABLES.items():
      count = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
print(f"{name:22} {count:>12,} rows")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_query_90d            2,414,248 rows


In [ ]:
# Step 2 — Check client history

clients = con.sql(f"""
    SELECT
            client_hash_id,
                    access_profile,
                            gsc_data_start,
                                    ga4_data_start
                                        FROM {TABLES['dim_clients']}
                                            ORDER BY gsc_data_start NULLS LAST
                                            """).df()
print("Number of clients:", len(clients))
print("\nFirst 10 clients:")
display(clients.head(10))

print("\nGSC data start:")
print(clients["gsc_data_start"].describe())

print("\nGA4 data start:")
print(clients["ga4_data_start"].describe())

Number of clients: 104

First 10 clients:


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19



GSC data start:
count                            67
mean     2025-11-17 00:42:59.104477
min             2025-01-27 00:00:00
25%             2025-09-24 00:00:00
50%             2025-11-05 00:00:00
75%             2026-02-19 00:00:00
max             2026-06-02 00:00:00
Name: gsc_data_start, dtype: object

GA4 data start:
count                            51
mean     2026-02-23 05:38:49.411764
min             2025-10-29 00:00:00
25%             2026-02-19 00:00:00
50%             2026-02-20 00:00:00
75%             2026-03-21 12:00:00
max             2026-06-01 00:00:00
Name: ga4_data_start, dtype: object


In [ ]:
# Week 3 — Verification Query 1
# Check the grain of the March 2026 daily fact table

grain_check = con.sql(f"""
    SELECT
            COUNT(*) AS total_rows,
                    COUNT(DISTINCT content_hash_id) AS unique_content_items,
                            COUNT(DISTINCT client_hash_id) AS unique_clients,
                                    COUNT(DISTINCT report_date) AS unique_dates
                                        FROM {TABLES['fact_daily']}
                                            WHERE report_date >= '2026-03-01'
                                                  AND report_date < '2026-04-01'
                                                  """).df()

display(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_content_items,unique_clients,unique_dates
0,9841378,331437,55,31


In [ ]:
# Week 3 — Verification Query 2
# Verify March 2026 row count and exact date span

march_panel = con.sql(f"""
    SELECT
            COUNT(*) AS row_count,
                    MIN(report_date) AS start_date,
                            MAX(report_date) AS end_date,
                                    COUNT(DISTINCT report_date) AS number_of_dates
                                        FROM {TABLES['fact_daily']}
                                            WHERE report_date >= '2026-03-01'
                                                  AND report_date < '2026-04-01'
                                                  """).df()

display(march_panel)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,start_date,end_date,number_of_dates
0,9841378,2026-03-01,2026-03-31,31


In [ ]:
# Week 3 — Verification Query 3
# Check availability of important fields in March 2026

availability_check = con.sql(f"""
    SELECT
            COUNT(*) AS total_rows,

                    COUNT(*) FILTER (
                                WHERE gsc_impressions IS TRUE
                                        ) AS rows_with_impressions,

                                                COUNT(*) FILTER (
                                                            WHERE gsc_clicks IS TRUE
                                                                    ) AS rows_with_clicks,

                                                                            COUNT(*) FILTER (
                                                                                        WHERE gsc_avg_position IS TRUE
                                                                                                ) AS rows_with_position

                                                                                                    FROM {TABLES['fact_daily']}
                                                                                                        WHERE report_date >= '2026-03-01'
                                                                                                              AND report_date < '2026-04-01'
                                                                                                              """).df()

display(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [ ]:
# Part 3 — Inspect content columns

content_columns = con.sql(f"""
    SELECT *
        FROM {TABLES['dim_content']}
            LIMIT 1
            """).df()

print("Columns in dim_content:")
print(content_columns.columns.tolist())

Columns in dim_content:
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


In [7]:
# Part 3 — Build the five-feature frame for March 2026
# Only include content that existed by the end of the March panel.

features_5 = con.sql(f"""
    SELECT
            client_hash_id,
                    content_hash_id,

                            -- Feature 1: search demand
                                    search_volume,

                                            -- Feature 2: keyword competition
                                                    competition,

                                                            -- Feature 3: content size
                                                                    word_count,

                                                                            -- Feature 4: content age at the end of March
                                                                                    DATE_DIFF(
                                                                                                'day',
                                                                                                            CAST(content_created_date AS DATE),
                                                                                                                        DATE '2026-03-31'
                                                                                                                                ) AS content_age_days,

                                                                                                                                        -- Feature 5: days since last update at the end of March
                                                                                                                                                DATE_DIFF(
                                                                                                                                                            'day',
                                                                                                                                                                        CAST(content_updated_date AS DATE),
                                                                                                                                                                                    DATE '2026-03-31'
                                                                                                                                                                                            ) AS days_since_last_update

                                                                                                                                                                                                FROM {TABLES['dim_content']}

                                                                                                                                                                                                    WHERE is_published IS TRUE
                                                                                                                                                                                                          AND is_deleted IS NOT TRUE
                                                                                                                                                                                                                AND CAST(content_created_date AS DATE) <= DATE '2026-03-31'
                                                                                                                                                                                                                      AND (
                                                                                                                                                                                                                                content_updated_date IS NULL
                                                                                                                                                                                                                                          OR CAST(content_updated_date AS DATE) <= DATE '2026-03-31'
                                                                                                                                                                                                                                                )
                                                                                                                                                                                                                                                """).df()
print(f"Feature rows: {len(features_5):,}")
display(features_5.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 42,669


,client_hash_id,content_hash_id,search_volume,competition,word_count,content_age_days,days_since_last_update
0,client_0797ff3a1fc9a6a5,content_04c67f3541177192,0,0.00,3168,175,34
1,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,20,0.03,4135,175,34
2,client_0797ff3a1fc9a6a5,content_0b33d8960857ad90,0,0.00,3232,175,34
3,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,0,0.00,3211,175,34
4,client_0797ff3a1fc9a6a5,content_167472cd0802a8f3,0,0.00,3149,175,34


### Feature validation

The initial feature calculation produced negative age values because some content records had dates after the March 2026 decision point. I corrected the feature construction by restricting content creation and update dates to information available on or before March 31, 2026.

After applying this prediction-time boundary, the feature frame contains 42,669 content items and the age/freshness values are non-negative.

This check reinforces the importance of defining a clear decision date before constructing features, because using information from after the decision point would introduce temporal leakage.

## 3. Five Features for the Prediction Task

For this first feature set, I selected five features that could be useful for identifying content pages that may be candidates for a future refresh.

| Feature | Why it matters | Available when? |
|---|---|---|
| `search_volume` | Represents the search demand associated with the content's target keyword. Higher demand means the page has more potential SEO value. | Available from content/keyword data before the prediction period. |
| `competition` | Indicates how competitive the keyword/search environment is. | Available from keyword data before the prediction period. |
| `word_count` | Represents the amount of content on the page and may capture differences in content depth. | Available from content metadata before the prediction period. |
| `content_age_days` | Represents how long the content has existed and may capture content lifecycle/freshness effects. | Calculated from the content creation date before the prediction period. |
| `days_since_last_update` | Represents how recently the content was updated and may provide a freshness signal. | Calculated from the last update date before the prediction period. |

### Feature-selection principle

I deliberately limited the first version to five features so that the model remains simple and interpretable.

The features are intended to represent information that could be known **before the future performance outcome is observed**. Performance metrics from the outcome window are not included because they could create data leakage.

### Feature Table Output

After applying the March 31, 2026 decision-time boundary, the feature frame contains 42,669 content items.

The feature table contains five selected features: `search_volume`, `competition`, `word_count`, `content_age_days`, and `days_since_last_update`.

The age and freshness calculations were validated to ensure that the features do not contain negative values caused by using information from after the decision point.

This reinforces the importance of defining a clear prediction-time boundary before building features.

## 4. Deliberate Leakage Experiment

Data leakage occurs when a model receives information that would not be available at the time a real prediction is made.

To demonstrate this, I will intentionally add a label-derived feature to the model inputs. This should produce an unrealistically strong score.

The leaked feature will then be removed, and the honest result will be retained.

This experiment is for demonstrating the leakage problem only; the leaked feature will not be used in the final model.

In [9]:
# Part 4 — Create a temporary label for the leakage demonstration

leak_test = features_5.copy()

# Temporary demonstration label.
# Missing search volume is treated as not meeting the condition.
leak_test["is_declining"] = (
    leak_test["search_volume"].fillna(0) < 10
    ).astype(int)

print("Label distribution:")
print(leak_test["is_declining"].value_counts())

Label distribution:
is_declining
0    23042
1    19627
Name: count, dtype: int64


In [15]:
# Part 4 — Create a proper temporary label for the leakage demonstration

leak_test = features_5.copy()

# Create a temporary label from content age.
# This is ONLY for demonstrating leakage.
leak_test["is_declining"] = (
    leak_test["content_age_days"] > 300
    ).astype(int)

print("Label distribution:")
print(leak_test["is_declining"].value_counts())

Label distribution:
is_declining
0    40061
1     2608
Name: count, dtype: int64


In [16]:
# Part 4 — Deliberately introduce label leakage

X_leaky = leak_test[[
    "search_volume",
        "competition",
            "word_count",
                "content_age_days",
                    "days_since_last_update",
                        "is_declining"       # <-- DELIBERATE LEAK
                        ]]

y = leak_test["is_declining"]

X_train, X_test, y_train, y_test = train_test_split(
X_leaky,
                  y,
                                    test_size=0.25,
                                        random_state=42,
                                            stratify=y
                                            )

leaky_model = DecisionTreeClassifier(random_state=42)
leaky_model.fit(X_train, y_train)

leaky_predictions = leaky_model.predict(X_test)

print(
                                                f"Accuracy with leakage: "
                                                    f"{accuracy_score(y_test, leaky_predictions):.3f}"
                                                    )

Accuracy with leakage: 1.000


In [17]:
# Part 4 — Remove the leaked information

honest_features = [
    "search_volume",
        "competition",
            "word_count",
                "days_since_last_update"
                ]

X_honest = leak_test[honest_features]
y = leak_test["is_declining"]

X_train, X_test, y_train, y_test = train_test_split(
                    X_honest,
                        y,
                            test_size=0.25,
                                random_state=42,
                                    stratify=y
                                    )

honest_model = DecisionTreeClassifier(random_state=42)
honest_model.fit(X_train, y_train)

honest_predictions = honest_model.predict(X_test)

print(
                                        f"Accuracy without leakage: "
                                            f"{accuracy_score(y_test, honest_predictions):.3f}"
                                            )

Accuracy without leakage: 0.974


### Leakage Experiment Result

The deliberately leaked model achieved an accuracy of 1.000 because the target variable `is_declining` was explicitly included among the model features.

After removing the leaked target and the feature used to construct the artificial label, the accuracy decreased to 0.974.

The difference demonstrates why label-derived information must not be included in the feature set. A perfect score can be caused by leakage rather than genuine predictive ability.

This experiment is a demonstration of the leakage mechanism only. The temporary `is_declining` label was artificially constructed for this exercise and is not treated as the final business target.

## 5. Limitation

A key limitation of this analysis is that the current feature table represents a snapshot of content metadata and historical information, but it does not contain a verified future refresh outcome.

Therefore, this notebook cannot establish whether refreshing a page will actually improve its performance.

The `is_declining` label used in the leakage experiment was artificially created only to demonstrate data leakage and is not a verified business outcome.

A future modelling stage should use a clearly defined prediction timeline and an outcome observed after the prediction point.

## 5. Self-check

- [x] I documented what one row represents for my lane.
- [x] I identified the warehouse table used for my analysis.
- [x] I used a mid-panel month rather than the final June 2026 sample for development.
- [x] I verified the grain, row count/date coverage, and data availability with SQL queries.
- [x] I checked availability using `IS TRUE`.
- [x] I selected exactly five initial features.
- [x] I documented why each feature is knowable at the decision moment.
- [x] I demonstrated label leakage deliberately.
- [x] I removed the leaked information and recorded the honest result.
- [x] I documented a limitation of the analysis.

### Final note

The feature set and leakage experiment are intended as a first data-contract exercise. The artificial label used for the leakage demonstration is not a verified business outcome. A future modelling stage requires a clearly defined prediction timeline and a genuine future outcome.